In [1]:
from pathlib import Path
import json, hashlib, uuid, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def check_daily(frame, date_col, start, end, label):
    idx = pd.DatetimeIndex(pd.to_datetime(frame[date_col]))
    expected = pd.date_range(start, end, freq='D')
    if idx.has_duplicates or not idx.equals(expected):
        raise ValueError(f'{label}: duplicate, missing or unexpected dates; '
                         f'expected {len(expected)} daily rows, got {len(idx)}. '
                         'Repair the source grid; missing observations are not assumed to be zero.')

SEGMENTS = {
    'SPLENDOR+':'100 CC', 'HF DELUXE':'100 CC', 'HF 100':'100 CC', 'PASSION':'100 CC',
    'GLAMOUR':'125 CC', 'SUPER SPLENDOR':'125 CC', 'XTREME 125':'125 CC',
    'XPULSE':'PREMIUM', 'XTREME 160':'PREMIUM', 'XTREME 250':'PREMIUM',
    'DESTINI':'SCOOTER', 'PLEASURE+':'SCOOTER', 'XOOM':'SCOOTER',
}


In [2]:
# Edit these settings before running. Use the same PROJECT_DIR in all three notebooks.
PROJECT_DIR = Path.cwd()
MODE = 'production'                 # 'production' or 'festive_backtest_2025'
POPULATION = 'all'              # 'series_a' or 'all'
SERIES_A_SELECTION = 'existing_table' # production: preserve existing membership
# 'recompute_global' reproduces the global top-85% concept with deterministic ties.
# 'recompute_by_model' selects top-85% within each model family (a different experiment).
TOP_SALES_SHARE = 0.85
MIN_TOP_MONTHS = 1
CHUNK_SIZE = 100
UTILS_DIR = Path('C:/Users/G0004878/Desktop/TFT_Data/utils_files')
TABLE_NAME = 'MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES'
VALID_TABLE = 'MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING'
TIME = 'CAL_DATE'
KEY = 'PARENT_DEALER_CODE_MODEL_FAMILY'
TARGET = 'NET_SALES'
DEALER = 'PARENT_DEALER_CODE'
TRAIN_START = '2023-04-01'
TRAIN_END = '2026-04-30'
VAL_START, VAL_END = '2026-05-01', '2026-08-31'
FC_START, FC_END = '2026-09-01', '2026-12-10'
ICL = 365
if MODE == 'festive_backtest_2025':
    TRAIN_END = '2025-04-30'
    VAL_START, VAL_END = '2025-05-01', '2025-08-31'
    FC_START, FC_END = '2025-09-01', '2025-12-10'
    if SERIES_A_SELECTION == 'existing_table':
        SERIES_A_SELECTION = 'recompute_global'
elif MODE != 'production':
    raise ValueError('Unknown MODE')
if POPULATION not in {'series_a', 'all'}:
    raise ValueError('Unknown POPULATION')
OCL = (pd.Timestamp(FC_END) - pd.Timestamp(FC_START)).days + 1
assert pd.Timestamp(VAL_END) + pd.Timedelta(days=1) == pd.Timestamp(FC_START)
assert pd.Timestamp(TRAIN_END) < pd.Timestamp(VAL_START) <= pd.Timestamp(VAL_END)
# One complete OCL-day holdout per series, using the final OCL days of validation.
VAL_OUTPUT_START = pd.Timestamp(VAL_END) - pd.Timedelta(days=OCL - 1)
assert VAL_OUTPUT_START >= pd.Timestamp(VAL_START)

STATIC = ['PARENT_DEALER_CODE','MODEL_FAMILY','BRAKE_TYPE',
          'IGNITION_TYPE','WHEEL_TYPE','COLOUR','DEALER_CITY',
          'X_CITY_CATEGORY','ZONAL_OFFICE_NAME']
FESTIVE = ['HARTALIK_TEEJ','GANESH_CHATURTHI','JANMASHTAMI','VISHWAKARMA_PUJA',
           'KARWA_CHAUTH','ONAM','HANUMAN_JAYANTI','AKSHYA_TRITIYA','BUDDHA_PURNIMA',
           'GANGA_DUSSEHRA','JAGANNATH_RATHYATRA','GURU_PURNIMA','NAG_PANCHAMI',
           'RAKSHA_BANDHAN','MARRIAGE_DAY','FESTIVE_DAYS_FROM_DIWALI',
           'IS_NAVRATARI','IS_NAVRATARI_START','IS_NAVRATARI_END','IS_DUSSEHRA',
           'IS_PITRA_PAKSHA','IS_DHANTERAS']
# Weight from true calendar offsets, not nonzero/clipped distance values.
FESTIVE_MULTIPLIER = 4.0             # 1 disables emphasis; 4 reproduces old 1+3 intent
WEIGHT_START_OFFSET, WEIGHT_END_OFFSET = -20, 10
# Optional explicit anchors if your distance feature is zero outside its active window.
# Use your source calendar's dates. Otherwise infer a UNIQUE zero per calendar year.
DIWALI_ANCHORS = {}                 # example format: {2026: 'YYYY-MM-DD'}

RUN_ID = datetime.now(timezone.utc).strftime('iteration3_%Y%m%d_%H%M%S_') + uuid.uuid4().hex[:8]
RUN_DIR = PROJECT_DIR / 'tft_runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'chunks').mkdir()
print('New data snapshot:', RUN_DIR)
print('Forecast:', FC_START, 'through', FC_END, '|', OCL, 'days; ICL:', ICL)


New data snapshot: c:\Users\G0004878\Desktop\_Iteraion_#4_all_series_updated_features\Modelling\tft_runs\iteration3_20260922_110627_f1cb5297
Forecast: 2026-09-01 through 2026-12-10 | 101 days; ICL: 365


In [3]:
import sys
sys.path.append(str(UTILS_DIR))
import Snowflake_configuration
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window

session = Session.builder.configs(Snowflake_configuration.ds1_role_json).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')
source = session.table(TABLE_NAME)
for old in source.columns:
    if old != old.replace('"', ''):
        source = source.rename(old, old.replace('"', ''))
required = list(dict.fromkeys([TIME, KEY, TARGET] + STATIC + FESTIVE))
missing = sorted(set(required) - set(source.columns))
if missing:
    raise ValueError(f'Missing source columns: {missing}')
source = source.select(*required).filter(F.col(TIME).between(F.lit(TRAIN_START), F.lit(FC_END)))

if POPULATION == 'all':
    selected = source.select(KEY).distinct()
elif SERIES_A_SELECTION == 'existing_table':
    selected = session.table(VALID_TABLE).select(KEY).distinct()
    print('Existing membership is frozen for this production snapshot; not used for historical backtests.')
elif SERIES_A_SELECTION in {'recompute_global', 'recompute_by_model'}:
    hist = source.filter(F.col(TIME) <= F.lit(TRAIN_END))
    month = hist.with_column('YM', F.date_trunc('MONTH', F.col(TIME)))
    monthly = month.group_by('YM', KEY, 'MODEL_FAMILY').agg(F.sum(TARGET).alias('SALES'))
    monthly = monthly.filter(F.col('SALES') > 0)
    parts = ['YM'] + (['MODEL_FAMILY'] if SERIES_A_SELECTION == 'recompute_by_model' else [])
    total_w = Window.partition_by(*parts)
    running_w = (Window.partition_by(*parts).order_by(F.col('SALES').desc(), F.col(KEY).asc())
                 .rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW))
    ranked = monthly.with_column('TOTAL', F.sum('SALES').over(total_w))
    ranked = ranked.with_column('RUNNING', F.sum('SALES').over(running_w))
    # Include the row crossing the threshold; explicit ROWS avoids tied-value RANGE jumps.
    top = ranked.filter((F.col('RUNNING') - F.col('SALES')) < TOP_SALES_SHARE * F.col('TOTAL'))
    selected = (top.group_by(KEY).agg(F.count('YM').alias('TOP_MONTHS'))
                .filter(F.col('TOP_MONTHS') >= MIN_TOP_MONTHS).select(KEY))
else:
    raise ValueError('Unknown SERIES_A_SELECTION')

# Collect once, then freeze membership for every subsequent query in this run.
selected_pd = selected.to_pandas()
if selected_pd.empty or selected_pd[KEY].isna().any() or selected_pd[KEY].duplicated().any():
    raise ValueError('Selected series IDs must be nonempty, unique and non-null')
selected_pd = selected_pd.sort_values(KEY).reset_index(drop=True)
selected_pd.to_parquet(RUN_DIR / 'selected_series.parquet', index=False)
selected_frozen = session.create_dataframe(selected_pd)
data = source.join(selected_frozen, on=KEY, how='inner')
# Preserve original IDs. Hash IDs only for filenames, preventing sanitization collisions.


In [4]:
# Build a genuinely shared calendar from all source rows, not an arbitrary dealer.
calendar = source.select(TIME, *FESTIVE).distinct().to_pandas()
calendar[TIME] = pd.to_datetime(calendar[TIME])
if calendar[TIME].duplicated().any():
    raise ValueError('Future covariates differ between series on the same date. '
                     'Regional/series-specific covariates need separate time series.')
calendar = calendar.sort_values(TIME).reset_index(drop=True)
check_daily(calendar, TIME, TRAIN_START, FC_END, 'Shared calendar')
for c in FESTIVE:
    calendar[c] = pd.to_numeric(calendar[c], errors='raise')
if not np.isfinite(calendar[FESTIVE].to_numpy(dtype=float)).all():
    raise ValueError('Calendar contains NaN or infinity')
binary = [c for c in FESTIVE if c != 'FESTIVE_DAYS_FROM_DIWALI']
if not calendar[binary].isin([0, 1]).all().all():
    raise ValueError('Expected 0/1 festival flags; inspect the source rather than casting silently')

def build_diwali_window(cal, date_col, anchors, left=-20, right=10):
    """Return a true calendar-offset mask; never derive it from clipped values."""
    if left > right:
        raise ValueError('Invalid Diwali window')
    dates = pd.DatetimeIndex(cal[date_col])
    mask = np.zeros(len(cal), dtype=bool)
    for year in sorted(set(dates.year)):
        explicit = anchors.get(year, anchors.get(str(year)))
        if explicit is None:
            candidates = dates[(dates.year == year) & cal['FESTIVE_DAYS_FROM_DIWALI'].eq(0).to_numpy()]
            if len(candidates) != 1:
                raise ValueError(f'{year}: expected one Diwali zero, got {len(candidates)}. '
                                 'Set DIWALI_ANCHORS from your source calendar; do not infer from clipped boundaries.')
            anchor = candidates[0]
        else:
            anchor = pd.Timestamp(explicit)
            if anchor.year != year:
                raise ValueError('Anchor year mismatch')
        offset = (dates - anchor).days
        mask |= (dates.year == year) & (offset >= left) & (offset <= right)
    return mask

def build_festive_weights(cal, multiplier, diwali_window):
    """Weight the Diwali window and Navratri; clipped distance is never used as a mask."""
    if multiplier < 1:
        raise ValueError('Invalid festive weighting configuration')
    mask = np.asarray(diwali_window, dtype=bool) | cal['IS_NAVRATARI'].eq(1).to_numpy()
    weights = np.where(mask, multiplier, 1.0).astype(np.float32)
    if multiplier > 1 and (not mask.any() or mask.all()):
        raise ValueError('Festive mask is empty or covers every date')
    return weights

calendar['IS_DIWALI_WINDOW'] = build_diwali_window(
    calendar, TIME, DIWALI_ANCHORS, WEIGHT_START_OFFSET, WEIGHT_END_OFFSET
).astype(np.float32)
calendar['WEIGHT'] = build_festive_weights(
    calendar, FESTIVE_MULTIPLIER, calendar['IS_DIWALI_WINDOW'].eq(1).to_numpy()
)
# Scale only the model input. IS_DIWALI_WINDOW and WEIGHT were already created
# from actual calendar dates, so clipping cannot cause distant dates to be weighted.
calendar['FESTIVE_DAYS_FROM_DIWALI'] = (
    calendar['FESTIVE_DAYS_FROM_DIWALI'].clip(WEIGHT_START_OFFSET, WEIGHT_END_OFFSET) / 20.0
).astype(np.float32)
dt = calendar[TIME].dt
calendar['DOW_SIN'] = np.sin(2*np.pi*dt.dayofweek/7)
calendar['DOW_COS'] = np.cos(2*np.pi*dt.dayofweek/7)
calendar['IS_MONTH_END'] = (dt.day >= dt.days_in_month-2).astype(float)
calendar['IS_MONTH_START'] = (dt.day <= 3).astype(float)
calendar['DAYS_TO_MONTH_END'] = (dt.days_in_month-dt.day)/31.0
FUTURE = FESTIVE + ['IS_DIWALI_WINDOW','DOW_SIN','DOW_COS',
                    'IS_MONTH_END','IS_MONTH_START','DAYS_TO_MONTH_END']
calendar[FUTURE] = calendar[FUTURE].astype(np.float32)
calendar.to_parquet(RUN_DIR / 'calendar.parquet', index=False)
print(calendar.groupby(calendar[TIME].dt.year)['WEIGHT'].agg(['min','max','mean']))


          min  max      mean
CAL_DATE                    
2023      1.0  4.0  1.414545
2024      1.0  4.0  1.311475
2025      1.0  4.0  1.312329
2026      1.0  4.0  1.331395


In [5]:
# Historical population coverage: same target definition, dates, model names and units.
def historical_totals(sdf):
    h = sdf.filter(F.col(TIME) <= F.lit(VAL_END))
    return (h.with_column('MONTH', F.date_trunc('MONTH', F.col(TIME)))
            .group_by('MODEL_FAMILY','MONTH').agg(F.sum(TARGET).alias('SALES')).to_pandas())
all_sales = historical_totals(source).rename(columns={'SALES':'ALL_SALES'})
a_sales = historical_totals(data).rename(columns={'SALES':'SELECTED_SALES'})
coverage = all_sales.merge(a_sales, on=['MODEL_FAMILY','MONTH'], how='left', validate='one_to_one')
coverage['SELECTED_SALES'] = coverage['SELECTED_SALES'].fillna(0)
coverage['SEGMENT'] = coverage.MODEL_FAMILY.map(SEGMENTS)
if coverage.SEGMENT.isna().any():
    raise ValueError('Update SEGMENTS for unrecognized model families')
coverage['RETAINED_SHARE'] = coverage.SELECTED_SALES / coverage.ALL_SALES.replace(0,np.nan)
coverage.to_csv(RUN_DIR / 'historical_coverage.csv', index=False)
print(coverage.groupby('SEGMENT')[['ALL_SALES','SELECTED_SALES']].sum())

dealers = sorted(data.select(DEALER).distinct().to_pandas()[DEALER].tolist(), key=str)
if not dealers or any(pd.isna(d) for d in dealers):
    raise ValueError('Missing dealers')
entries, total_rows = [], 0
pull_columns = list(dict.fromkeys([TIME, KEY, TARGET] + STATIC))
for i in range(0, len(dealers), CHUNK_SIZE):
    batch = dealers[i:i+CHUNK_SIZE]
    # Production never needs unknown future targets. Backtest targets are evaluation-only.
    end = FC_END if MODE == 'festive_backtest_2025' else VAL_END
    frame = (data.filter(F.col(DEALER).isin(batch)).filter(F.col(TIME) <= F.lit(end))
             .select(*pull_columns).to_pandas())
    if frame.empty:
        raise ValueError('Empty dealer chunk')
    frame[TIME] = pd.to_datetime(frame[TIME])
    if frame[[KEY]+STATIC].isna().any().any():
        raise ValueError('Null IDs/static values: fix or explicitly encode upstream')
    values = pd.to_numeric(frame[TARGET], errors='raise').to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or (values<0).any() or not np.equal(values,np.rint(values)).all():
        raise ValueError('Negative-binomial target must be finite, nonnegative integer counts. No silent clamping.')
    if values.max() >= 2**24:
        raise ValueError('Target exceeds exact float32 integer range')
    frame[TARGET] = values.astype(np.float32)
    frame = frame.sort_values([KEY,TIME]).reset_index(drop=True)
    path = RUN_DIR / 'chunks' / f'chunk_{i//CHUNK_SIZE:04d}.parquet'
    frame.to_parquet(path, index=False)
    entries.append({'path':str(path.relative_to(RUN_DIR)), 'sha256':digest(path), 'rows':len(frame)})
    total_rows += len(frame)
    print(f'Chunk {len(entries)}: {len(frame):,} rows')

config = dict(schema_version=1, run_id=RUN_ID, mode=MODE, population=POPULATION,
              selection=SERIES_A_SELECTION,
              selection_cutoff=TRAIN_END if POPULATION=='series_a' and SERIES_A_SELECTION.startswith('recompute') else None,
              top_sales_share=TOP_SALES_SHARE, min_top_months=MIN_TOP_MONTHS,
              source_table=TABLE_NAME, time_col=TIME, group_col=KEY, target_col=TARGET,
              static_covariates=STATIC, future_covariates=FUTURE,
              train_start=TRAIN_START, train_end=TRAIN_END, val_start=VAL_START, val_end=VAL_END,
              forecast_start=FC_START, forecast_end=FC_END, icl=ICL, ocl=OCL,
              val_output_start=str(VAL_OUTPUT_START.date()), festive_multiplier=FESTIVE_MULTIPLIER,
              weight_offsets=[WEIGHT_START_OFFSET,WEIGHT_END_OFFSET],
              explicit_diwali_anchors=DIWALI_ANCHORS, segments=SEGMENTS)
write_json(RUN_DIR / 'config.json', config)
write_json(RUN_DIR / 'data_manifest.json', {'chunks':entries, 'rows':total_rows,
    'config_hash':digest(RUN_DIR/'config.json'), 'calendar_hash':digest(RUN_DIR/'calendar.parquet'),
    'membership_hash':digest(RUN_DIR/'selected_series.parquet')})
(RUN_DIR / 'DATA_READY').write_text('complete', encoding='utf-8')
write_json(PROJECT_DIR / 'iteration3_active_run.json', {'run_dir':str(RUN_DIR.resolve())})
session.close()
print('Data ready:', RUN_DIR)
print('Next: run modelling_code_iteration3_updated.ipynb')


          ALL_SALES  SELECTED_SALES
SEGMENT                            
100 CC   15418293.0      15418293.0
125 CC    1834860.0       1834860.0
PREMIUM    176444.0        176444.0
SCOOTER   1147795.0       1147795.0
Chunk 1: 12,096,565 rows
Chunk 2: 12,849,712 rows
Chunk 3: 12,692,338 rows
Chunk 4: 12,601,161 rows
Chunk 5: 13,063,291 rows
Chunk 6: 12,358,855 rows
Chunk 7: 12,913,411 rows
Chunk 8: 12,659,864 rows
Chunk 9: 12,524,972 rows
Chunk 10: 11,404,619 rows
Chunk 11: 4,058,001 rows
Chunk 12: 781,874 rows
Data ready: c:\Users\G0004878\Desktop\_Iteraion_#4_all_series_updated_features\Modelling\tft_runs\iteration3_20260922_110627_f1cb5297
Next: run modelling_code_iteration3_updated.ipynb
